# Giftmaxxing MTL recommender — train & deploy

End-to-end SageMaker workflow for the feed value model:
`Score = 2·P_Time + 5·P_Custom + 1·P_Buy` over (user taste centroid, Titan Multimodal item vector).

**Where to run this**
- **Notebook instance:** `ml.t3.medium` is plenty (dataset ≈ 2k examples, few MB; no GPU — the model is a tiny MLP). Kernel: `conda_pytorch_p310` or any PyTorch ≥ 2.x kernel. SageMaker Studio works identically.
- **Execution role IAM:** `dynamodb:Scan` on `giftmaxxing-dev-*`, `s3vectors:ListVectors|GetVectors` on the `pins` index, read/write on `s3://giftmaxxing-dev-ml`, plus the standard `AmazonSageMakerFullAccess` for Training Jobs / endpoints.
- **Local dev:** everything here also runs from `infra/ml/` with the repo venv (`.venv`) and `AWS_PROFILE=dev_sso_giftmaxxing` — the SageMaker-specific cells (§4–§6) are the only parts that need the SDK/role.

Companion scripts live one directory up: `export_training_data.py`, `train.py`, `inference.py`, `mtl_model.py`, `cluster_gifts.py`.


In [ ]:
import sys, subprocess, json, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sagemaker', 'boto3', 'scikit-learn', 'matplotlib'], check=True)
import boto3, sagemaker
REGION      = 'us-east-1'
ML_BUCKET   = 'giftmaxxing-dev-ml'          # datasets + model artifacts
ENDPOINT    = 'giftmaxxing-dev-mtl'         # serverless endpoint name (Lambda's MTL_ENDPOINT)
sess  = sagemaker.Session(boto3.Session(region_name=REGION))
role  = sagemaker.get_execution_role() if 'SM_' in str(os.environ) or os.path.exists('/opt/ml') else os.environ.get('SAGEMAKER_ROLE_ARN', '')
print('role:', role)


## 1 · Export the training data
Joins DynamoDB analytics (impressions + `dwellMs`) and interactions with the S3 Vectors Titan embeddings into `(user, post)` examples with the three labels. Group-split by user. Uploads to `s3://giftmaxxing-dev-ml/datasets/mtl/<stamp>/`.


In [ ]:
!python ../export_training_data.py --out ../data --s3-bucket {ML_BUCKET}


## 2 · EDA — label balance & dwell distribution


In [ ]:
import numpy as np, matplotlib.pyplot as plt
meta = json.load(open('../data/meta.json')); print(json.dumps(meta, indent=2)[:600])
tr = np.load('../data/train.npz', allow_pickle=True)
labels = meta['labels']
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].bar(labels, tr['Y'].sum(axis=0)); ax[0].set_title('train positives per head')
ax[1].hist(tr['X_aux'][:, 0], bins=40); ax[1].set_title('cos(user, item) over examples')
plt.tight_layout()


## 3 · Train locally (seconds — sanity check before paying for a job)


In [ ]:
!python ../train.py --data ../data --out ../model
print(open('../model/metrics.json').read())


## 4 · SageMaker Training Job (the managed path)
Same `train.py`, unchanged — it reads `SM_CHANNEL_TRAIN`/`SM_MODEL_DIR`. Use the newest dataset stamp printed by §1. `ml.m5.large` spot ≈ pennies per run.


In [ ]:
from sagemaker.pytorch import PyTorch
DATASET = sorted(sess.list_s3_files(ML_BUCKET, 'datasets/mtl/'))[-1].rsplit('/', 1)[0]  # newest stamp
est = PyTorch(entry_point='train.py', source_dir='..', role=role,
              framework_version='2.3', py_version='py311',
              instance_count=1, instance_type='ml.m5.large',
              use_spot_instances=True, max_wait=3600, max_run=1800,
              hyperparameters={'epochs': 200, 'patience': 20},
              output_path=f's3://{ML_BUCKET}/models/mtl/')
est.fit({'train': f's3://{ML_BUCKET}/{DATASET}/'})


## 5 · Register the model (Model Registry)
Versioned + approvable; the deploy cell below can take either the estimator or an approved package.


In [ ]:
pkg = est.register(model_package_group_name='giftmaxxing-mtl',
                   content_types=['application/json'], response_types=['application/json'],
                   inference_instances=['ml.m5.large'], approval_status='PendingManualApproval')
print(pkg.model_package_arn)


## 6 · Deploy — SERVERLESS endpoint (scales to zero, pennies at our traffic)
`inference.py` handles the JSON contract the `/recommendations` Lambda sends. Cold start for the PyTorch container is 10–30 s; the Lambda treats a timeout as a soft failure and falls back to cosine order, so cold hits warm the endpoint without hurting users.


In [ ]:
from sagemaker.pytorch import PyTorchModel
from sagemaker.serverless import ServerlessInferenceConfig
model = PyTorchModel(model_data=est.model_data, role=role,
                     entry_point='inference.py', source_dir='..',
                     framework_version='2.3', py_version='py311')
predictor = model.deploy(serverless_inference_config=ServerlessInferenceConfig(
                             memory_size_in_mb=2048, max_concurrency=5),
                         endpoint_name=ENDPOINT)
print('endpoint:', ENDPOINT)


## 7 · Smoke-test the endpoint


In [ ]:
import numpy as np
va = np.load('../data/val.npz', allow_pickle=True)
payload = {'user': va['X_user'][0].tolist(), 'user_events': 40,
           'items': [{'key': k.split('|')[1], 'vector': v.tolist(), 'price': 25, 'source': 'shopify'}
                     for k, v in zip(va['keys'][:5], va['X_item'][:5])]}
smr = boto3.client('sagemaker-runtime', region_name=REGION)
resp = smr.invoke_endpoint(EndpointName=ENDPOINT, ContentType='application/json', Body=json.dumps(payload))
print(json.dumps(json.loads(resp['Body'].read()), indent=1))


## 8 · Gift grouping — cluster the Titan vectors
The image-understanding pipeline's output doubles as the gift-pool generator (CLOUD.md §15.3). Read-only by default; add `--apply` to publish `pool#<id>` rows.


In [ ]:
!python ../cluster_gifts.py --k 24 --out ../clusters.json
c = json.load(open('../clusters.json'))
for cl in c['clusters'][:12]: print(f"#{cl['clusterId']:>2} n={cl['size']:<5} {cl['label']}")


## 9 · Wire it to the API
Set the Lambda env `MTL_ENDPOINT=giftmaxxing-dev-mtl` (terraform `-var mtl_endpoint=giftmaxxing-dev-mtl` → apply). `/recommendations` then re-ranks its kNN candidates through this endpoint and returns `source: "vector+mtl"` with per-item `p_time/p_custom/p_buy/mtlScore`.

## Cleanup (endpoint has no idle cost, but to remove entirely)


In [ ]:
# predictor.delete_endpoint(); sagemaker.Session().delete_model(model.name)
